In [ ]:
%load_ext autoreload
%autoreload 2

from spatial_tcr.utils import setup_plotting, switch_cwd_to_root

switch_cwd_to_root()

figure_dir = "figures/revision"
setup_plotting(figure_dir, display_dpi=300, save_dpi=300)

import os

import numpy as np
import scanpy as sc
import scipy

from spatial_tcr.tcr import get_tcr_genes
from spatial_tcr.utils import (
    aggregate_gene_expression,
)

## Load data

In [7]:
path = "data/xenium/processed/05.2-kidney_tcr_infilrate.h5ad"
adata = sc.read_h5ad(path)
adata.X = adata.layers["counts"].copy()

In [8]:
adata.var_names

Index(['ABCC11', 'ACE2', 'ACKR1', 'ACTA2', 'ADAMTS1', 'ADGRE1', 'ADGRL4',
       'ADH1C', 'ADH4', 'ADIPOQ',
       ...
       'TYROBP', 'UMOD', 'VCAM1', 'VIM', 'VSIG4', 'VWA5A', 'VWF', 'XCL1',
       'XCL2', 'ZNF683'],
      dtype='object', length=480)

## Filtering after BLAST analysis

In [9]:
bv_map_old = {
    "TRBV10": ["TRBV10-1", "TRBV10-2", "TRBV10-3"],
    "TRBV11": ["TRBV11-1", "TRBV11-2"],
    "TRBV12": ["TRBV12-3", "TRBV12-4", "TRBV12-5"],
    "TRBV18_19": ["TRBV18", "TRBV19"],
    "TRBV4": ["TRBV4-1", "TRBV4-2"],
    "TRBV5-6": ["TRBV5-3", "TRBV5-5", "TRBV5-6", "TRBV5-7"],
    "TRBV6": ["TRBV6-1", "TRBV6-2", "TRBV6-5", "TRBV6-7"],
    "TRBV7-2_3": ["TRBV7-2", "TRBV7-3"],
}  # "TRBV7-6":["TRBV7-4","TRBV7-6","TRBV7-7"]
av_map_old = {
    "TRAV19_21": ["TRAV19", "TRAV21"],
    "TRAV12": ["TRAV12-1", "TRAV12-2", "TRAV12-3"],
    "TRAV8-4": ["TRAV8-2", "TRAV8-3", "TRAV8-4", "TRAV8-5", "TRAV8-6"],
    "TRAV9": ["TRAV9-1", "TRAV9-2"],
}

removed_trvgenes_old = [
    "TRBV12-2",
    "TRBV17",
    "TRBV2",
    "TRBV22-1",
    "TRBV26",
    "TRBV6-4",
    "TRBV7-1",
    "TRBV7-5",
    "TRBV8-2",
    "TRBVA",
    "TRAV11",
    "TRAV15",
    "TRAV28",
    "TRAV30",
    "TRAV31",
    "TRAV32",
    "TRAV33",
    "TRAV37",
    "TRAV8-7",
    "TRBV20-1",
    "TRAV38-2DV8",
    "TRAV4",
]

In [10]:
bv_map = {
    "TRBV10": ["TRBV10-1", "TRBV10-2", "TRBV10-3"],
    "TRBV11": ["TRBV11-1", "TRBV11-2"],
    "TRBV12": ["TRBV12-3", "TRBV12-4", "TRBV12-5"],
    "TRBV18_19": ["TRBV18", "TRBV19"],
    "TRBV4": ["TRBV4-1", "TRBV4-2"],
    "TRBV5-6": ["TRBV5-3", "TRBV5-5", "TRBV5-6", "TRBV5-7", "TRBV5-4"],
    "TRBV6": ["TRBV6-1", "TRBV6-2", "TRBV6-5", "TRBV6-7", "TRBV6-8"],
    "TRBV7-2_3": ["TRBV7-2", "TRBV7-3"],
    "TRBV7-6": ["TRBV7-4", "TRBV7-6", "TRBV7-7"],
}

av_map = {
    "TRAV19_21": ["TRAV19", "TRAV21"],
    "TRAV12": ["TRAV12-1", "TRAV12-2", "TRAV12-3"],
    "TRAV8-4": ["TRAV8-2", "TRAV8-3", "TRAV8-4", "TRAV8-5", "TRAV8-6"],
    "TRAV9": ["TRAV9-1", "TRAV9-2"],
    # "TRAV1-1": {"TRAV1-1", "TRAV1-2"},
}

removed_trvgenes = [
    "TRBV12-2",
    "TRBV17",
    "TRBV2",
    "TRBV22-1",
    "TRBV26",
    "TRBV6-4",
    "TRBV7-1",
    "TRBV7-5",
    "TRBV8-2",
    "TRBVA",
    "TRAV11",
    "TRAV15",
    "TRAV28",
    "TRAV30",
    "TRAV31",
    "TRAV32",
    "TRAV33",
    "TRAV37",
    "TRAV8-7",
    "TRBV20-1",
    "TRAV38-2DV8",
    "TRAV4",
    "TRBV29-1",
]


avbv_map = {**av_map, **bv_map}

In [11]:
new_removed_trvgenes = [g for g in removed_trvgenes if g not in removed_trvgenes_old]
del_removed_trvgenes = [g for g in removed_trvgenes_old if g not in removed_trvgenes]
print(new_removed_trvgenes)
print(del_removed_trvgenes)

['TRBV29-1']
[]


In [12]:
adata = adata[:, ~adata.var_names.isin(removed_trvgenes)].copy()

In [13]:
ad_merged = aggregate_gene_expression(adata, avbv_map)

In [14]:
ad_merged.layers["counts"] = ad_merged.X.copy()

In [15]:
tv_genes = get_tcr_genes(ad_merged)[-1]

Found 35 TRAV genes, 31 TRBV genes, 3 TRDV genes, 14 TRGV genes


In [9]:
# mask trv expression for all non-tcells
X_mask = np.ones(ad_merged.shape, dtype=bool)
tcell_mask = (ad_merged.obs["cell_type_no_tcr"] == "T").values[:, None]
print(X_mask.shape)
X_mask = tcell_mask * X_mask

X_mask[:, 0].sum(), tcell_mask[:, 0].sum()

gene_mask = ~ad_merged.var_names.isin(tv_genes)

X_mask[:, gene_mask] = True

ad_merged.layers["counts_clean"] = (
    ad_merged.layers["counts"]
    .multiply(scipy.sparse.csr_matrix(X_mask, dtype=int))
    .tocsr()
)

(510139, 431)


In [10]:
non_tv_genes = [v for v in ad_merged.var_names if v not in tv_genes]
ad_merged[:, non_tv_genes].layers["counts_clean"] != ad_merged[:, non_tv_genes].layers[
    "counts"
]

<Compressed Sparse Row sparse matrix of dtype 'bool'
	with 0 stored elements and shape (510139, 348)>

In [11]:
ad_merged[ad_merged.obs["cell_type_no_tcr"] != "T", tv_genes].layers[
    "counts_clean"
].sum()

np.float64(0.0)

In [12]:
data_dir = "data/xenium/processed"
os.makedirs(data_dir, exist_ok=True)
ad_merged.write_h5ad(f"{data_dir}/06.2-kidney_tcr_filtered.h5ad")

/home/dschaub/.uv-local/venvs/xenium-tcr/lib/python3.13/site-packages/anndata/_io/utils.py:243: FutureWarning: Forward slashes will be disallowed in h5 stores in the next minor release
  return func(*args, **kwargs)
